# Hugging Face tokenizers で BPE / WordPiece を学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_subword/tokenizers_train.ipynb)

NLP クックブック 05-03 ／ サブワードとトークナイザ

Transformers でそのまま使えるトークナイザを、自分のコーパスから高速に学習したい。

In [1]:
import nltk
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg

lines = [l for l in gutenberg.raw("austen-emma.txt").split("\n") if l.strip()]
print(len(lines), "行")

14283 行


In [2]:
from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainers

tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=3000, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], show_progress=False)
tok.train_from_iterator(lines, trainer)

enc = tok.encode("Emma Woodhouse was unquestionably handsome.")
print(enc.tokens)
print(enc.ids)
print(tok.decode(enc.ids))

['Ġemma', 'Ġwoodhouse', 'Ġwas', 'Ġun', 'qu', 'est', 'ion', 'ably', 'Ġhandsome', '.']
[187, 368, 113, 307, 176, 351, 124, 728, 1423, 12]
 emma woodhouse was unquestionably handsome.


In [3]:
# Transformers のトークナイザとして保存・読み込み
from transformers import AutoTokenizer, PreTrainedTokenizerFast

fast = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token="[UNK]", pad_token="[PAD]", bos_token="[BOS]", eos_token="[EOS]")
fast.save_pretrained("my_tokenizer")
print(AutoTokenizer.from_pretrained("my_tokenizer")("Hello Emma!"))

{'input_ids': [76, 85, 42, 187, 4], 'attention_mask': [1, 1, 1, 1, 1]}


- WordPiece にするなら `models.WordPiece` + `trainers.WordPieceTrainer` + `pre_tokenizers.Whitespace()` に差し替える。
- 既存モデルと同じ方式で作り直すだけなら `old_tokenizer.train_new_from_iterator(lines, vocab_size)` が最短。